# River-1 quickstart

Use River-1 to explain synthetic weekly sales and forecast the eight weeks after them. The same inference and plot calls work with a local bundle, Hugging Face, or the hosted River API.

Install PyMC MFM on Python 3.13 or newer with `pip install pymc-mfm` (or `uv add pymc-mfm`, or `conda install -c conda-forge pymc-mfm`). The one install covers a local bundle, Hugging Face, and the hosted API; there are no extras.

## 1. Make a history and a future plan

The history has 104 weeks of sales, TV and search spend, and price. The future plan has the same spend and control columns, but no sales. Its dates begin one week after the history ends. The numbers are synthetic and do not describe a real business.

In [ ]:
import numpy as np
import pandas as pd

import pymc_mfm as mfm

rng = np.random.default_rng(0)
weeks = pd.date_range("2024-11-11", periods=104, freq="7D")
tv = rng.gamma(4.0, 2_000.0, size=104)
search = rng.gamma(4.0, 1_500.0, size=104)
price = rng.normal(10.0, 0.3, size=104)
noise = rng.normal(0, 500, size=104)
sales = 40_000 + 0.8 * tv + 1.2 * search - 1_000 * (price - 10.0) + noise
history = pd.DataFrame({"week": weeks, "sales": sales, "tv": tv, "search": search, "price": price})
future = pd.DataFrame(
    {
        "week": pd.date_range(weeks[-1] + pd.Timedelta(weeks=1), periods=8, freq="7D"),
        "tv": 8_000.0,
        "search": 6_000.0,
        "price": 10.0,
    }
)

## 2. Load a model

The cell below runs the pinned River-1 bundle from Hugging Face on this machine. Change only the `mfm.load(...)` line to select where River runs:

- **River-1 from Hugging Face** (below): `mfm.load("river-1", **roles)`. The weights are private to the PyMC Labs Hugging Face organization, so this needs PyMC Labs Hub access: use `hf auth login`, `HF_TOKEN`, or a Hugging Face `token=`.
- **Bundle on disk:** `mfm.load("./models/my-model", **roles)`, with the path to your bundle directory. No credentials are needed.
- **Hosted:** `mfm.load("river-1", engine="hosted", **roles)` needs an invitation key. Set `MFM_API_URL` and `MFM_API_KEY` from the River API console's Connect page, or pass the key as `token=`. Keep it separate from your Hugging Face credentials and out of this notebook. Inference sends the history and plan to River and downloads results, not model weights. Use the synthetic data unless your invitation allows your own.

In [ ]:
roles = {
    "date_column": "week",
    "target_column": "sales",
    "channel_columns": ["tv", "search"],
    "control_columns": ["price"],
}
model = mfm.load("river-1", **roles)

## 3. Run inference

`model.inference(context, query)` takes the history table as `context` and the weeks to explain or forecast as `query`. River uses the supplied data without fitting new model weights. This example uses 100 draws for a quick look; the default is 1,000. `result.idata` keeps the history (`context`) and forecast (`query`) in separate groups. Equal draw indices do not make a joint trajectory across the two windows.

In [ ]:
result = model.inference(history, future, n_samples=100, random_seed=0)
result.idata

## 4. Plot the result

Inspect the input series, check expected sales against observed sales in the history, and see which components contribute to the expected outcome. The shares chart summarizes each channel's or control's share. The intervals exclude observation noise, and contributions are not evidence of causal lift.

The charts read the completed result without running inference again. `model.plot` is also available for charts of the latest result.

In [ ]:
inputs = result.plot.inputs()
inputs

In [ ]:
fit = result.plot.fit()
fit

In [ ]:
contributions = result.plot.contributions_over_time()
contributions

In [ ]:
shares = result.plot.shares()
shares